# Time Series Analysis — UK Macroeconomic Variables
---

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

from fredapi import Fred

import matplotlib.pyplot as plt

from scipy import stats
from scipy.special import inv_boxcox
from scipy.stats import yeojohnson

from pmdarima.arima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Data Loading
---

In [ ]:
os.chdir('/Users/dannyhogan/Desktop/ST-498')
df_UK = pd.read_csv("Data/MacroVariablesUK.csv", index_col=0)
df_US = pd.read_csv("Data/MacroVariablesUS.csv", index_col=0)

# Automated SARIMA Pipeline
---
Runs the full transformation + model-selection workflow across every column in `df_UK`.
Results are stored in `results` (best model per column) and `comparisons` (full comparison table).

In [ ]:
os.chdir('/Users/dannyhogan/Desktop/ST-498/Code')
from SARIMA_Fitting import run_sarima_pipeline, fit_all_transformations, forecast_best_model

results, comparisons = run_sarima_pipeline(df_UK, m=4, n_periods=8)

# Manual Model Exploration
---
Deep-dive into specific series to validate transformation choices and inspect diagnostics.

## GDP (Real GDP)

### Data Preparation

In [ ]:
UK_GDP = df_UK[['rGDP']].copy()

### Model Selection
Fit `auto_arima` on the original series and two common transformations, then compare diagnostics.

#### Original Series

In [ ]:
model_gdp_raw = auto_arima(
    y=UK_GDP['rGDP'],
    start_p=0, start_q=0,
    m=4,
    information_criterion='aic'
)
print(model_gdp_raw.summary())

#### Box-Cox Transformation

In [ ]:
UK_GDP['BoxCox'], lam_gdp_bc = stats.boxcox(UK_GDP['rGDP'])

model_gdp_bc = auto_arima(
    y=UK_GDP['BoxCox'],
    start_p=0, start_q=0,
    start_P=0, start_Q=0,
    m=4,
    information_criterion='aic'
)
print(model_gdp_bc.summary())

#### Log Transformation

In [ ]:
UK_GDP['Log'] = np.log(UK_GDP['rGDP'])

model_gdp_log = auto_arima(y=UK_GDP['Log'], m=4, seasonal=True)
print(model_gdp_log.summary())

### Findings
---
The log-transformed model significantly outperforms both the Box-Cox and raw series by AIC.
The `auto_arima`-selected parameters from the log model will be used for the final forecast.

### Forecast

In [ ]:
forecast_log      = model_gdp_log.predict(n_periods=8)
forecast_from_log = np.exp(forecast_log)

forecast_index = pd.date_range(start=UK_GDP.index[-1], periods=9, freq='QE')[1:]

fig, ax = plt.subplots(figsize=(14, 8))
recent_data = UK_GDP['rGDP'].iloc[-40:]

ax.plot(recent_data.index, recent_data.values, label='Historical', linewidth=2)
ax.plot(forecast_index, forecast_from_log,
        label='Forecast (Log Transform)',
        linewidth=2, color='purple', marker='o', markersize=8)
ax.set_title('UK GDP Forecast — Log Transformation')
ax.set_ylabel('Real GDP')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Growth Rate Analysis

In [ ]:
growth_rates = (np.diff(forecast_from_log) / forecast_from_log[:-1]) * 100

print("Quarter-to-Quarter Growth Rates (Forecast):")
for i, rate in enumerate(growth_rates, 1):
    print(f"  Q{i} -> Q{i+1}: {rate:.3f}%")

print(f"\nGrowth rate std deviation: {np.std(growth_rates):.5f}%")
print("(Low std = nearly constant growth rate = exponential trend)")

In [ ]:
UK_GDP['QoQ_Growth'] = UK_GDP['rGDP'].pct_change() * 100

print("Historical Quarterly Growth Rates:")
for label, window in [("Last 5 years  (20 quarters)", 20), ("Last 10 years (40 quarters)", 40)]:
    subset = UK_GDP['QoQ_Growth'].iloc[-window:]
    print(f"\n{label}:")
    print(f"  Mean: {subset.mean():.3f}%")
    print(f"  Std:  {subset.std():.3f}%")
    print(f"  Min:  {subset.min():.3f}%")
    print(f"  Max:  {subset.max():.3f}%")

Growth is reasonable as a baseline for UK GDP, sitting slightly above the 10-year quarterly average.
A few points to keep in mind going forward:

- **Forecast convergence:** Predicted growth converges to a near-constant after a few quarters. For a 3–5 year horizon this warrants attention.
- **Exogenous variables:** Adding drivers such as interest rates, inflation, or trade data via SARIMAX could materially improve the forecast.
- **Low forecast variance:** Acceptable for a baseline scenario, but will need revisiting when modelling alternative market conditions.

## CPI (Inflation)

### Data Preparation

In [ ]:
CPI = df_UK[['cpi']].copy()
CPI = CPI.dropna()                   # drop any trailing NaN (e.g., incomplete 2026 obs.)
CPI['cpi'] = CPI['cpi'] + 1e-4      # small offset required for Yeo-Johnson

### Transformations

In [ ]:
CPI['log']                    = np.log(CPI['cpi'])
CPI['YeoJohnson'], opt_lambda = yeojohnson(CPI['cpi'])

### Model Comparison
Fit `auto_arima` on each transformation and review the full SARIMAX diagnostics.

In [ ]:
for col in CPI.columns:
    model_col = auto_arima(y=CPI[col], m=4, seasonal=True)
    print(f"{'='*55}")
    print(f"  {col}")
    print(f"{'='*55}")
    print(model_col.summary())
    print()

### Forecast
The Yeo-Johnson transformation returns the best diagnostic scores.
Refit and produce a 16-quarter (4-year) forecast.

In [ ]:
yeo = auto_arima(y=CPI['YeoJohnson'], m=4, seasonal=True)

In [ ]:
def inv_yeojohnson(y, lmbda):
    """Inverse Yeo-Johnson transformation."""
    y   = np.asarray(y)
    inv = np.zeros_like(y)
    pos = y >= 0

    if lmbda != 0:
        inv[pos] = np.power(y[pos] * lmbda + 1, 1 / lmbda) - 1
    else:
        inv[pos] = np.exp(y[pos]) - 1

    if lmbda != 2:
        inv[~pos] = 1 - np.power(-(2 - lmbda) * y[~pos] + 1, 1 / (2 - lmbda))
    else:
        inv[~pos] = 1 - np.exp(-y[~pos])

    return inv


forecast_yeo      = yeo.predict(n_periods=16)
forecast_from_yeo = inv_yeojohnson(forecast_yeo, opt_lambda)

if not isinstance(CPI.index, pd.DatetimeIndex):
    CPI.index = pd.to_datetime(CPI.index)

last_date      = CPI.index[-1]
forecast_index = pd.date_range(
    start   = last_date + pd.DateOffset(months=3),
    periods = 16,
    freq    = 'QE'
)

fig, ax = plt.subplots(figsize=(14, 8))
recent_data = CPI['cpi'].iloc[-40:]

ax.plot(recent_data.index, recent_data.values, label='Historical', linewidth=2)
ax.plot(forecast_index, forecast_from_yeo,
        label='Forecast (Yeo-Johnson Transform)',
        linewidth=2, color='purple', marker='o', markersize=8)
ax.set_title('UK CPI Forecast — Yeo-Johnson Transformation')
ax.set_ylabel('CPI')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()